# 可视化系统

本教程演示 HIcosmo 的统一可视化系统（基于 GetDist 后端）。

**关键 API**：
- `Plotter(chain_name)` — 从链名创建绘图器
- `Plotter.from_fisher(mean, cov, names)` — 从 Fisher 矩阵创建
- `plotter.corner()` — 角图（Corner Plot）
- `plotter.plot_1d()` / `plotter.plot_2d()` — 1D/2D 分布
- `plotter.traces()` — 链轨迹图
- `plotter.report()` — 统计报告（含 AIC/BIC）

In [ ]:
import hicosmo as hc
hc.init()

## 1. 准备数据：运行 MCMC

In [ ]:
from hicosmo.samplers import MCMC
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood
from hicosmo.models import LCDM

# 参数配置
params = {
    'H0': (70.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

# SNe 链
sne = SN_likelihood(LCDM, "pantheon+")
mcmc_sn = MCMC(params, sne, chain_name='viz_sn')
mcmc_sn.run(num_samples=2000)

# BAO 链
bao = BAO_likelihood(LCDM, "desi2024")
mcmc_bao = MCMC(params, bao, chain_name='viz_bao')
mcmc_bao.run(num_samples=2000)

## 2. Plotter 类

In [ ]:
from hicosmo.visualization import Plotter

# 方法1: 从链名创建（最简）
plotter = Plotter('viz_sn')

# 方法2: 从字典创建
samples = {'H0': mcmc_sn.samples['H0'], 'Omega_m': mcmc_sn.samples['Omega_m']}
plotter_dict = Plotter(samples)

# 方法3: 自定义标签和范围
plotter_custom = Plotter(
    'viz_sn',
    labels={'H0': r'$H_0\,[\mathrm{km/s/Mpc}]$'},
    ranges={'H0': (65, 75)}
)

## 3. 角图（Corner Plot）

In [ ]:
# 基本角图
plotter.corner(['H0', 'Omega_m'], filename='figures/08_corner.pdf')

# 绘制所有参数
plotter.corner(filename='figures/08_corner_all.pdf')

## 4. 1D/2D 分布图

In [ ]:
# 1D 边缘分布
plotter.plot_1d('H0', filename='figures/08_1d_H0.pdf')

# 2D 等高线图
plotter.plot_2d('H0', 'Omega_m', filename='figures/08_2d_contour.pdf')

# 填充等高线
plotter.plot_2d('H0', 'Omega_m', filled=True, filename='figures/08_2d_filled.pdf')

## 5. 多链比较

In [ ]:
# 多链 Plotter
plotter_multi = Plotter(
    ['viz_sn', 'viz_bao'],
    labels=['Pantheon+ SNe', 'DESI BAO']
)

# 多链角图
plotter_multi.corner(['H0', 'Omega_m'], filename='figures/08_multi_corner.pdf')

# 多链 1D 比较
plotter_multi.plot_1d('H0', filename='figures/08_multi_1d.pdf')

# 多链 2D 比较
plotter_multi.plot_2d('H0', 'Omega_m', filename='figures/08_multi_2d.pdf')

## 6. 链轨迹图

In [ ]:
# 链轨迹（诊断收敛性）
plotter.traces(['H0', 'Omega_m'], filename='figures/08_traces.pdf')

## 7. 统计报告

In [ ]:
# 打印统计报告（含 AIC/BIC/Evidence）
plotter.report()

## 8. Fisher 预测可视化

In [ ]:
import numpy as np

# 从 Fisher 矩阵创建 Plotter
mean = [67.36, 0.3153]
covariance = [[0.36, -0.005], [-0.005, 0.0004]]

plotter_fisher = Plotter.from_fisher(
    mean=mean,
    covariance=covariance,
    param_names=['H0', 'Omega_m'],
    nsample=10000
)

# Fisher 角图
plotter_fisher.corner(['H0', 'Omega_m'], filename='figures/08_fisher_corner.pdf')

## 9. 样式和颜色

In [ ]:
from hicosmo.visualization import apply_style
from hicosmo.visualization.plotting import MODERN_COLORS, CLASSIC_COLORS

# 应用专业样式
apply_style()

print(f"Modern 配色: {MODERN_COLORS}")
print(f"Classic 配色: {CLASSIC_COLORS}")

## API 速查

```python
from hicosmo.visualization import Plotter, apply_style

# === 创建 Plotter ===
plotter = Plotter('chain_name')                    # 从链名
plotter = Plotter(['chain1', 'chain2'], labels=['A', 'B'])  # 多链
plotter = Plotter({'H0': array, 'Omega_m': array})  # 从字典
plotter = Plotter.from_fisher(mean, cov, names)    # 从 Fisher

# === 绘图方法 ===
plotter.corner(['H0', 'Omega_m'], filename='corner.pdf')  # 角图
plotter.corner()  # 所有参数

plotter.plot_1d('H0', filename='1d.pdf')           # 1D 分布
plotter.plot_2d('H0', 'Omega_m', filename='2d.pdf')  # 2D 等高线
plotter.plot_2d('H0', 'Omega_m', filled=True)      # 填充等高线

plotter.traces(['H0', 'Omega_m'], filename='traces.pdf')  # 链轨迹

# === 统计报告 ===
plotter.report()  # 打印统计信息（含 AIC/BIC/Evidence）

# === 自定义 ===
plotter = Plotter(
    'chain',
    labels={'H0': r'$H_0$ [km/s/Mpc]'},  # LaTeX 标签
    ranges={'H0': (65, 75)},              # 显示范围
    style='modern'                        # 'modern' 或 'classic'
)

# === 样式 ===
apply_style()  # 应用专业绘图样式
```

### 数据输入格式

| 格式 | 示例 | 说明 |
|------|------|------|
| 字符串 | `'my_chain'` | 链名称（自动加载 .h5 文件） |
| 列表 | `['chain1', 'chain2']` | 多链比较 |
| 字典 | `{'H0': arr, ...}` | 样本数据 |
| Fisher | `.from_fisher(mean, cov, names)` | Fisher 预测 |